# Encoders in Transformers

# Import Libraries

In [1]:
import torch
import torch.nn as nn
import math

# Multi-Head Self-Attention

In [2]:
# ---------------------------------------------------------
# Multi-Head Self-Attention
# ---------------------------------------------------------

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear Layers to Project input into Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # Linear layer after concatenating heads
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.size()

        # Project Input → Q, K, V
        Q = self.W_q(x)  # (B, L, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # Split into heads
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product attention
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, heads, L, L)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn = scores.softmax(dim=-1)  # (B, heads, L, L)

        output = attn @ V  # (B, heads, L, d_k)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        return self.W_o(output)  # (B, L, d_model)


# Feed Forward Network

In [3]:
# ---------------------------------------------------------
# Feed-Forward Network (position-wise)
# ---------------------------------------------------------
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.layer1 = nn.Linear(d_model, d_ff)
        self.layer2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))


# Transformer Encoder Block

In [4]:
# ---------------------------------------------------------
# Transformer Encoder Block (Pre-LN version)
# ---------------------------------------------------------
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadSelfAttention(d_model, num_heads)
        self.dropout1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # ---- Pre-LN Self-Attention ----
        norm_x = self.ln1(x)
        attn_output = self.self_attn(norm_x, mask)
        x = x + self.dropout1(attn_output)  # residual

        # ---- Pre-LN Feed-Forward ----
        norm_x = self.ln2(x)
        ffn_output = self.ffn(norm_x)
        x = x + self.dropout2(ffn_output)  # residual

        return x  # (B, L, d_model)


# Test Encoder Block

In [6]:
# ---------------------------------------------------------
# Test the Encoder Block
# ---------------------------------------------------------
if __name__ == "__main__":
    batch = 2
    seq_len = 20
    d_model = 512

    x = torch.randn(batch, seq_len, d_model)
    layer = TransformerEncoderLayer(d_model=512, num_heads=8, d_ff=2048)

    output = layer(x)
    print("Output shape:", output.shape)

Output shape: torch.Size([2, 20, 512])


# Send Text

In [8]:
# === 1. SIMPLE TOKENIZER (toy example) ===
vocab = {
    "how": 1,
    "are": 2,
    "you": 3,
    "<unk>": 0
}

def tokenize(text):
    return [vocab.get(word, 0) for word in text.lower().split()]


# === 2. BUILD EMBEDDING + POSITIONAL ENCODING ===
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=50):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, token_ids):
        seq_len = token_ids.size(1)
        positions = torch.arange(0, seq_len).unsqueeze(0)

        return self.token_emb(token_ids) + self.pos_emb(positions)


# === 3. USE YOUR ENCODER LAYER ===
encoder_layer = TransformerEncoderLayer(d_model=512)

# === 4. SEND "how are you" ===
text = "how are you"
token_ids = tokenize(text)
token_ids = torch.tensor([token_ids])     # shape: (batch=1, seq_len=3)

embedder = InputEmbedding(vocab_size=10000, d_model=512)
input_vectors = embedder(token_ids)       # shape: (1, 3, 512)

output = encoder_layer(input_vectors)
print("Output:", output.shape)

Output: torch.Size([1, 3, 512])


# ✔ How to Stack 6 Encoder Layers (like original Transformer)

In [7]:
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x